In [ ]:
# --- Imports, device, reproducibility, and global settings ---

import gc
import json
import math
import os
import random
import time
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from tqdm.auto import trange

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

try:
    BF16_OK = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
except Exception:
    BF16_OK = False
AMP_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print("AMP_DTYPE:", AMP_DTYPE)

TRAIN_ITERS = 30_000
EFFECTIVE_BATCH_SIZE = 4096
WARMUP_ITERS = 200
LEARNING_RATE = 2e-3
MIN_LR = 1e-6
GRAD_CLIP_NORM = None
LOG_INTERVAL = 500

N_HEADS = 1
ALLOW_DUPLICATES = False

EARLY_STOP_SAMPLE_ACCURACY_THRESHOLD = 0.9999 # should probably set to none if trying to analyze length generalization (so that you can keep training past 100% accuracy)
SEED_CHOICES = [SEED]
WEIGHT_DECAY_CHOICES = [0, 0.01, 0.1]
BLOCK_SIZE_CHOICES = [16]
TRAIN_LENGTH_METHOD_CHOICES = ["fixed"] # other options include: "random_range", "curriculum_up", "pure_up_curriculum", "two_part_curriculum", "curriculum_down"
USE_MLP_CHOICES = [False]
DEPTH_CHOICES = [1]
VOCAB_N_CHOICES = [64]
EMBEDDING_RATIO_CHOICES = [("N", 1)]
USE_POSITIONAL_EMBEDDING_CHOICES = [True]
FINAL_LN_CHOICES = [True]

EVAL_DURING_TRAINING = False # can set to true if you want to monitor length generalization as a function of training steps
LENGTH_GENERALIZATION_EVALS_ENABLED = (EARLY_STOP_SAMPLE_ACCURACY_THRESHOLD is None)
EVAL_LENGTH_MULTIPLIERS = [0.5, 1, 2, 3, 4, 5, 6]
LENGTH_EVAL_BATCH_SIZE = 64
POST_TRAINING_EVAL_BATCHES = 3

GRID_LOOP_ORDER_INNER_TO_OUTER = [
    "seed",
    "train_length_method",
    "weight_decay",
    "use_final_ln",
    "use_positional_embedding",
    "embedding_ratio",
    "vocab_n",
    "depth",
    "use_mlp",
    "block_size",
]

expected_models = (
    len(SEED_CHOICES) * len(WEIGHT_DECAY_CHOICES) * len(FINAL_LN_CHOICES)
    * len(USE_POSITIONAL_EMBEDDING_CHOICES) * len(EMBEDDING_RATIO_CHOICES)
    * len(VOCAB_N_CHOICES) * len(DEPTH_CHOICES) * len(USE_MLP_CHOICES)
    * len(TRAIN_LENGTH_METHOD_CHOICES) * len(BLOCK_SIZE_CHOICES)
)

print("Grid loop order (inner -> outer):", " -> ".join(GRID_LOOP_ORDER_INNER_TO_OUTER))
print("Expected number of models:", expected_models)
print("Length-generalization evals enabled:", LENGTH_GENERALIZATION_EVALS_ENABLED)
print("Eval during training:", EVAL_DURING_TRAINING and LENGTH_GENERALIZATION_EVALS_ENABLED)
print("Eval length multipliers:", EVAL_LENGTH_MULTIPLIERS)


In [34]:
# --- Shared helpers: batching, model, grid construction, training, saving, evaluation ---

def _is_oom(e: BaseException) -> bool:
    msg = str(e).lower()
    return ("out of memory" in msg) or ("cuda oom" in msg) or ("cublas" in msg and "alloc" in msg)


def make_generator(device: torch.device, seed: int) -> torch.Generator:
    try:
        g = torch.Generator(device=device.type)
    except Exception:
        g = torch.Generator()
    g.manual_seed(int(seed))
    return g


def autocast_ctx(device: torch.device, enabled: bool = True):
    if (not enabled) or device.type != "cuda":
        return nullcontext()
    try:
        return torch.amp.autocast("cuda", dtype=AMP_DTYPE)
    except Exception:
        return torch.cuda.amp.autocast(dtype=AMP_DTYPE)


def make_grad_scaler(enabled: bool):
    if not enabled:
        class _NoScaler:
            def is_enabled(self): return False
            def scale(self, x): return x
            def step(self, opt): opt.step()
            def update(self): pass
            def unscale_(self, opt): pass
        return _NoScaler()
    try:
        return torch.amp.GradScaler()
    except Exception:
        return torch.cuda.amp.GradScaler()


def float_token(value: float) -> str:
    return str(value).replace("-", "m").replace(".", "p")


def sanitize_filename(text: str) -> str:
    return "".join(ch if (ch.isalnum() or ch in ("-", "_", ".")) else "_" for ch in str(text))


def sample_uniform_int(low: int, high: int, generator: torch.Generator) -> int:
    if low > high:
        raise ValueError(f"Invalid integer range: [{low}, {high}]")
    if low == high:
        return int(low)
    return int(torch.randint(int(low), int(high) + 1, (1,), device=DEVICE, generator=generator).item())


def resolve_eval_lengths(base_k: int, vocab_n: int, multipliers: Sequence[float]) -> List[int]:
    lengths: List[int] = []
    seen = set()
    for mult in multipliers:
        length = max(1, int(round(float(base_k) * float(mult))))
        if length > int(vocab_n) or length in seen:
            continue
        seen.add(length)
        lengths.append(length)
    return lengths


def select_train_length(cfg: "TrainConfig", itr: int, generator: torch.Generator) -> int:
    k = int(cfg.block_size)
    if cfg.train_length_method == "fixed":
        return k
    if cfg.train_length_method == "random_range":
        return sample_uniform_int(1, k, generator)
    progress = float(itr / max(cfg.max_iters - 1, 1))
    if cfg.train_length_method == "curriculum_up":
        current_max = min(k, max(1, 1 + int(progress * (k - 1))))
        return sample_uniform_int(1, current_max, generator)
    if cfg.train_length_method == "pure_up_curriculum":
        return min(k, max(1, 1 + int(progress * (k - 1))))
    if cfg.train_length_method == "two_part_curriculum":
        halfway = max(cfg.max_iters // 2, 1)
        if itr < halfway:
            first_half_progress = float(itr / max(halfway - 1, 1))
            return min(k, max(1, 1 + int(first_half_progress * (k - 1))))
        return sample_uniform_int(1, k, generator)
    if cfg.train_length_method == "curriculum_down":
        current_min = max(1, min(k, k - int(progress * (k - 1))))
        return sample_uniform_int(current_min, k, generator)
    raise ValueError(f"Unknown train_length_method: {cfg.train_length_method}")


def _sample_numbers(batch_size: int, vocab_n: int, length: int, device: torch.device, allow_duplicates: bool, *, generator: Optional[torch.Generator] = None) -> torch.Tensor:
    if allow_duplicates:
        return torch.randint(0, vocab_n, (batch_size, length), device=device, generator=generator, dtype=torch.long)
    if length > vocab_n:
        raise ValueError(f"allow_duplicates=False requires length <= vocab_n (got {length} > {vocab_n})")
    scores = torch.rand(batch_size, vocab_n, device=device, generator=generator)
    return scores.topk(length, dim=1).indices.to(torch.long)


def get_batch(batch_size: int, length: int, device: torch.device, *, vocab_n: int, allow_duplicates: bool, generator: Optional[torch.Generator] = None) -> torch.Tensor:
    x = _sample_numbers(batch_size=batch_size, vocab_n=vocab_n, length=length, device=device, allow_duplicates=allow_duplicates, generator=generator)
    vals = x.sort(dim=1).values
    sep = torch.full((batch_size, 1), vocab_n, device=device, dtype=torch.long)
    return torch.cat([x, sep, vals], dim=1)


class MLP(nn.Module):
    def __init__(self, n_embd: int):
        super().__init__()
        self.fc_1 = nn.Linear(n_embd, 3 * n_embd)
        self.gelu = nn.GELU(approximate="tanh")
        self.fc_2 = nn.Linear(3 * n_embd, n_embd)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc_2(self.gelu(self.fc_1(x)))


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_heads: int, n_layers: int):
        super().__init__()
        assert n_embd % n_heads == 0
        self.n_embd = int(n_embd)
        self.n_heads = int(n_heads)
        self.head_dim = int(n_embd // n_heads)
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)


class Block(nn.Module):
    def __init__(self, n_embd: int, n_heads: int, n_layers: int, use_mlp: bool = True):
        super().__init__()
        self.attn = CausalSelfAttention(n_embd=n_embd, n_heads=n_heads, n_layers=n_layers)
        self.ln_1 = nn.LayerNorm(n_embd)
        self.use_mlp = bool(use_mlp)
        if self.use_mlp:
            self.mlp = MLP(n_embd)
            self.ln_2 = nn.LayerNorm(n_embd)
        else:
            self.mlp = None
            self.ln_2 = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))
        if self.mlp is not None:
            x = x + self.mlp(self.ln_2(x))
        return x


@dataclass
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layers: int
    n_heads: int
    n_embd: int
    without_pos: bool
    use_mlp: bool
    use_final_LN: bool
    max_seq_len: int


class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.n_layers = int(config.n_layers)
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.max_seq_len, config.n_embd),
            h=nn.ModuleList([Block(config.n_embd, config.n_heads, config.n_layers, use_mlp=config.use_mlp) for _ in range(config.n_layers)]),
            ln_f=(nn.LayerNorm(config.n_embd) if config.use_final_LN else nn.Identity()),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight
        self.apply(self._init_weights)
        self.register_buffer("pos_idx", torch.arange(config.max_seq_len), persistent=False)
        if config.without_pos:
            with torch.no_grad():
                self.transformer.wpe.weight.zero_()
            self.transformer.wpe.weight.requires_grad_(False)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.xavier_uniform_(module.weight)

    def forward(self, idx: torch.Tensor, *, block_size: int, return_full_logits: bool = False) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T = idx.size()
        expected_T = 2 * int(block_size) + 1
        assert T == expected_T, f"Expected T={expected_T}, got T={T}"
        assert T <= self.config.max_seq_len, f"T={T} exceeds max_seq_len={self.config.max_seq_len}"
        pos = self.transformer.wpe(self.pos_idx[:T])
        x = self.transformer.wte(idx) if self.config.without_pos else (self.transformer.wte(idx) + pos)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits_half = self.lm_head(x[:, block_size:T - 1, :])
        targets = idx[:, block_size + 1:]
        loss = F.cross_entropy(logits_half.reshape(-1, logits_half.size(-1)), targets.reshape(-1))
        if return_full_logits:
            return self.lm_head(x), loss
        return logits_half, loss


@dataclass
class TrainConfig:
    train_order: int
    model_id: str
    block_size: int
    train_length_method: str
    vocab_n: int
    n_layers: int
    n_heads: int
    n_embd: int
    emb_ratio_label: str
    without_pos: bool
    use_mlp: bool
    use_final_LN: bool
    max_seq_len: int
    max_supported_length: int
    allow_duplicates: bool = ALLOW_DUPLICATES
    max_iters: int = TRAIN_ITERS
    effective_batch_size: int = EFFECTIVE_BATCH_SIZE
    warmup_iters: int = WARMUP_ITERS
    learning_rate: float = LEARNING_RATE
    min_lr: float = MIN_LR
    weight_decay: float = 0.0
    grad_clip_norm: Optional[float] = GRAD_CLIP_NORM
    log_interval: int = LOG_INTERVAL
    early_stop_sample_accuracy_threshold: Optional[float] = EARLY_STOP_SAMPLE_ACCURACY_THRESHOLD
    seed: int = SEED
    save_dir: str = "./final_models"
    eval_dir: str = "./length_generalization"
    eval_during_training: bool = bool(EVAL_DURING_TRAINING and LENGTH_GENERALIZATION_EVALS_ENABLED)
    eval_length_multipliers: Tuple[float, ...] = tuple(EVAL_LENGTH_MULTIPLIERS)
    eval_batch_size: int = LENGTH_EVAL_BATCH_SIZE
    post_training_eval_batches: int = POST_TRAINING_EVAL_BATCHES


GRID_DIMENSION_NAMES = ["seed", "weight_decay", "block_size", "train_length_method", "use_mlp", "depth", "vocab_n", "embedding_ratio", "use_positional_embedding", "use_final_ln"]


def validate_grid_loop_order(inner_to_outer: List[str]) -> None:
    missing = [name for name in GRID_DIMENSION_NAMES if name not in inner_to_outer]
    extra = [name for name in inner_to_outer if name not in GRID_DIMENSION_NAMES]
    duplicates = [name for name in inner_to_outer if inner_to_outer.count(name) > 1]
    if missing or extra or duplicates or len(inner_to_outer) != len(GRID_DIMENSION_NAMES):
        raise ValueError("GRID_LOOP_ORDER_INNER_TO_OUTER must contain each grid dimension exactly once. " f"missing={missing}, extra={extra}, duplicates={sorted(set(duplicates))}")


def iter_grid_choices(dimension_name: str):
    if dimension_name == "seed": return SEED_CHOICES
    if dimension_name == "weight_decay": return WEIGHT_DECAY_CHOICES
    if dimension_name == "block_size": return BLOCK_SIZE_CHOICES
    if dimension_name == "train_length_method": return TRAIN_LENGTH_METHOD_CHOICES
    if dimension_name == "use_mlp": return USE_MLP_CHOICES
    if dimension_name == "depth": return DEPTH_CHOICES
    if dimension_name == "vocab_n": return VOCAB_N_CHOICES
    if dimension_name == "embedding_ratio": return EMBEDDING_RATIO_CHOICES
    if dimension_name == "use_positional_embedding": return USE_POSITIONAL_EMBEDDING_CHOICES
    if dimension_name == "use_final_ln": return FINAL_LN_CHOICES
    raise KeyError(f"Unknown grid dimension: {dimension_name}")


def build_experiment_specs(*, save_dir: str, eval_dir: str) -> List[TrainConfig]:
    validate_grid_loop_order(GRID_LOOP_ORDER_INNER_TO_OUTER)
    out: List[TrainConfig] = []
    train_order = 1
    loop_order_outer_to_inner = list(reversed(GRID_LOOP_ORDER_INNER_TO_OUTER))

    def add_configs(selected: Dict[str, Any], depth_idx: int) -> None:
        nonlocal train_order
        if depth_idx == len(loop_order_outer_to_inner):
            seed = int(selected["seed"])
            weight_decay = float(selected["weight_decay"])
            block_size = int(selected["block_size"])
            train_length_method = str(selected["train_length_method"])
            use_mlp = bool(selected["use_mlp"])
            n_layers = int(selected["depth"])
            vocab_n = int(selected["vocab_n"])
            emb_ratio_label, emb_ratio = selected["embedding_ratio"]
            n_embd = max(1, int(round(vocab_n * float(emb_ratio))))
            use_pos_emb = bool(selected["use_positional_embedding"])
            use_final_LN = bool(selected["use_final_ln"])
            eval_lengths = resolve_eval_lengths(block_size, vocab_n, EVAL_LENGTH_MULTIPLIERS)
            max_supported_length = max([block_size] + eval_lengths)
            model_id = (
                f"sortgpt_k{block_size}_meth{train_length_method}_mlp{int(use_mlp)}_L{n_layers}"
                f"_N{vocab_n}_E{n_embd}_pos{int(use_pos_emb)}_fln{int(use_final_LN)}"
                f"_wd{float_token(weight_decay)}_seed{seed}"
            )
            out.append(TrainConfig(
                train_order=train_order,
                model_id=model_id,
                block_size=block_size,
                train_length_method=train_length_method,
                vocab_n=vocab_n,
                n_layers=n_layers,
                n_heads=int(N_HEADS),
                n_embd=int(n_embd),
                emb_ratio_label=str(emb_ratio_label),
                without_pos=not use_pos_emb,
                use_mlp=use_mlp,
                use_final_LN=use_final_LN,
                max_seq_len=int(2 * max_supported_length + 1),
                max_supported_length=int(max_supported_length),
                seed=seed,
                weight_decay=weight_decay,
                save_dir=str(save_dir),
                eval_dir=str(eval_dir),
            ))
            train_order += 1
            return
        dimension_name = loop_order_outer_to_inner[depth_idx]
        for choice in iter_grid_choices(dimension_name):
            selected[dimension_name] = choice
            add_configs(selected, depth_idx + 1)

    add_configs({}, 0)
    return out


def final_model_name(cfg: TrainConfig) -> str:
    return f"{cfg.model_id}__final.pt"


def final_model_path(cfg: TrainConfig) -> Path:
    return Path(cfg.save_dir) / final_model_name(cfg)


def during_training_plot_path(cfg: TrainConfig) -> Path:
    return Path(cfg.eval_dir) / "during_training" / f"{sanitize_filename(cfg.model_id)}__during_training.png"


def post_training_eval_csv_path() -> Path:
    return Path(POST_TRAINING_EVAL_DIR) / "length_generalization_eval.csv"


def post_training_eval_plot_path() -> Path:
    return Path(POST_TRAINING_EVAL_DIR) / "length_generalization_eval.png"


def create_optimizer(model: nn.Module, *, weight_decay: float, lr: float) -> torch.optim.Optimizer:
    params = [p for p in model.parameters() if p.requires_grad]
    if DEVICE.type == "cuda":
        try:
            return torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.95), eps=1e-8, weight_decay=float(weight_decay), fused=True)
        except Exception:
            pass
    return torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.95), eps=1e-8, weight_decay=float(weight_decay))


def get_lr(itr: int, cfg: TrainConfig) -> float:
    if itr < cfg.warmup_iters:
        return cfg.learning_rate * (itr + 1) / (cfg.warmup_iters + 1)
    if itr >= cfg.max_iters:
        return cfg.min_lr
    ratio = (itr - cfg.warmup_iters) / max(cfg.max_iters - cfg.warmup_iters, 1)
    ratio = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return cfg.min_lr + ratio * (cfg.learning_rate - cfg.min_lr)


@torch.no_grad()
def evaluate_teacher_forced_lengths(model: nn.Module, cfg: TrainConfig, lengths: Sequence[int], *, num_batches: int, generator_seed: int) -> Dict[int, Dict[str, float]]:
    was_training = model.training
    model.eval()
    out: Dict[int, Dict[str, float]] = {}
    for length in lengths:
        token_correct = 0
        token_total = 0
        loss_values: List[float] = []
        generator = make_generator(DEVICE, int(generator_seed + 97 * int(length)))
        for _ in range(int(num_batches)):
            idx = get_batch(batch_size=int(cfg.eval_batch_size), length=int(length), device=DEVICE, vocab_n=int(cfg.vocab_n), allow_duplicates=bool(cfg.allow_duplicates), generator=generator)
            with autocast_ctx(DEVICE, enabled=True):
                logits, loss = model(idx, block_size=int(length), return_full_logits=False)
            targets = idx[:, int(length) + 1:]
            preds = logits.detach().argmax(dim=-1)
            token_correct += int((preds == targets).sum().item())
            token_total += int(targets.numel())
            loss_values.append(float(loss.detach().float().item()))
        out[int(length)] = {
            "token_acc": float(token_correct / max(token_total, 1)),
            "loss": float(sum(loss_values) / max(len(loss_values), 1)),
            "num_batches": int(num_batches),
        }
    if was_training:
        model.train()
    return out


def save_training_eval_plot(cfg: TrainConfig, history: List[Dict[str, Any]]) -> Optional[str]:
    rows: List[Dict[str, Any]] = []
    eval_lengths = resolve_eval_lengths(cfg.block_size, cfg.vocab_n, cfg.eval_length_multipliers)
    for row in history:
        metrics = row.get("eval_token_acc_by_length")
        if not metrics:
            continue
        for length in eval_lengths:
            if int(length) not in metrics:
                continue
            rows.append({"iter": int(row["iter"]), "eval_length": int(length), "token_acc": float(metrics[int(length)])})
    if not rows:
        return None
    df = pd.DataFrame(rows)
    plot_path = during_training_plot_path(cfg)
    plot_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(10, 6))
    for length, group in df.groupby("eval_length"):
        group = group.sort_values("iter")
        plt.plot(group["iter"], group["token_acc"], marker="o", linewidth=1.5, label=f"L={length}")
    plt.xlabel("Training steps")
    plt.ylabel("Token accuracy")
    plt.title(f"During-training length generalization\\n{cfg.model_id}")
    plt.grid(alpha=0.25)
    plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
    plt.tight_layout()
    plt.savefig(plot_path, dpi=180, bbox_inches="tight")
    plt.close()
    return str(plot_path)


def save_summary_csv(rows_by_id: Dict[str, Dict[str, Any]], path: Path) -> pd.DataFrame:
    path.parent.mkdir(parents=True, exist_ok=True)
    columns = [
        "train_order", "model_id", "artifact_name", "seed", "block_size", "train_length_method",
        "weight_decay", "use_mlp", "n_layers", "vocab_n", "model_vocab_size", "embedding_size",
        "embedding_ratio", "use_final_LN", "allow_duplicates", "without_pos", "n_heads", "max_iters",
        "max_seq_len", "max_supported_length", "eval_during_training", "eval_length_multipliers",
        "training_time_minutes", "final_training_loss", "final_training_token_accuracy",
        "final_training_sample_accuracy", "iter_to_sample_acc_gt_0_9999", "during_training_eval_plot_path",
        "artifact_path",
    ]
    df = pd.DataFrame(list(rows_by_id.values()))
    if len(df) == 0:
        df = pd.DataFrame(columns=columns)
    else:
        df = df.sort_values(["train_order", "model_id"]).reset_index(drop=True)
        for col in columns:
            if col not in df.columns:
                df[col] = np.nan
        df = df[columns]
    df.to_csv(path, index=False)
    return df


def save_run_manifest(path: Path, *, run_dir: Path, models_dir: Path, summary_csv_path: Path, experiments: List[TrainConfig]) -> None:
    payload = {
        "created_at": datetime.utcnow().isoformat() + "Z",
        "run_dir": str(run_dir),
        "models_dir": str(models_dir),
        "summary_csv_path": str(summary_csv_path),
        "num_models": int(len(experiments)),
        "global_settings": {
            "seed": int(SEED),
            "seed_choices": [int(x) for x in SEED_CHOICES],
            "train_iters": int(TRAIN_ITERS),
            "effective_batch_size": int(EFFECTIVE_BATCH_SIZE),
            "warmup_iters": int(WARMUP_ITERS),
            "learning_rate": float(LEARNING_RATE),
            "min_lr": float(MIN_LR),
            "weight_decay_choices": [float(x) for x in WEIGHT_DECAY_CHOICES],
            "grad_clip_norm": GRAD_CLIP_NORM,
            "n_heads": int(N_HEADS),
            "train_length_method_choices": list(TRAIN_LENGTH_METHOD_CHOICES),
            "embedding_ratio_choices": [(str(label), float(ratio)) for label, ratio in EMBEDDING_RATIO_CHOICES],
            "use_positional_embedding_choices": [bool(x) for x in USE_POSITIONAL_EMBEDDING_CHOICES],
            "early_stop_sample_accuracy_threshold": EARLY_STOP_SAMPLE_ACCURACY_THRESHOLD,
            "allow_duplicates": bool(ALLOW_DUPLICATES),
            "length_generalization_evals_enabled": bool(LENGTH_GENERALIZATION_EVALS_ENABLED),
            "eval_during_training": bool(EVAL_DURING_TRAINING and LENGTH_GENERALIZATION_EVALS_ENABLED),
            "eval_length_multipliers": [float(x) for x in EVAL_LENGTH_MULTIPLIERS],
            "eval_batch_size": int(LENGTH_EVAL_BATCH_SIZE),
            "post_training_eval_batches": int(POST_TRAINING_EVAL_BATCHES),
        },
        "experiments": [asdict(cfg) for cfg in experiments],
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


def load_existing_summary_row(artifact_path: Path) -> Optional[Dict[str, Any]]:
    if not artifact_path.exists():
        return None
    artifact = torch.load(artifact_path, map_location="cpu")
    if not isinstance(artifact, dict):
        return None
    row = artifact.get("summary_row")
    return row if isinstance(row, dict) else None


def load_model_from_artifact(artifact_path: Path) -> Tuple[nn.Module, Dict[str, Any], TrainConfig]:
    artifact = torch.load(artifact_path, map_location="cpu")
    model_cfg = GPTConfig(**artifact["model_config"])
    train_cfg = TrainConfig(**artifact["train_config"])
    model = GPT(model_cfg)
    model.load_state_dict(artifact["model_state_dict"])
    model = model.to(DEVICE)
    model.eval()
    return model, artifact, train_cfg


def train_sortgpt(cfg: TrainConfig, *, force_retrain: bool = False) -> Dict[str, Any]:
    artifact_path = final_model_path(cfg)
    artifact_path.parent.mkdir(parents=True, exist_ok=True)
    if artifact_path.exists() and not force_retrain:
        row = load_existing_summary_row(artifact_path)
        if row is not None:
            print(f"[skip existing final model] {artifact_path.name}")
            return row

    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    random.seed(cfg.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(cfg.seed)

    train_generator = make_generator(DEVICE, cfg.seed)
    model_cfg = GPTConfig(
        block_size=int(cfg.block_size),
        vocab_size=int(cfg.vocab_n + 1),
        n_layers=int(cfg.n_layers),
        n_heads=int(cfg.n_heads),
        n_embd=int(cfg.n_embd),
        without_pos=bool(cfg.without_pos),
        use_mlp=bool(cfg.use_mlp),
        use_final_LN=bool(cfg.use_final_LN),
        max_seq_len=int(cfg.max_seq_len),
    )
    model = GPT(model_cfg).to(DEVICE)
    scaler = make_grad_scaler(enabled=(DEVICE.type == "cuda" and AMP_DTYPE == torch.float16))
    optimizer = create_optimizer(model, weight_decay=cfg.weight_decay, lr=cfg.learning_rate)

    history: List[Dict[str, Any]] = []
    iter_to_sample_acc_gt_0_9999: Optional[int] = None
    eval_lengths = resolve_eval_lengths(cfg.block_size, cfg.vocab_n, cfg.eval_length_multipliers)

    print(
        f"[train] order={cfg.train_order:03d} | seed={cfg.seed} | k={cfg.block_size} | method={cfg.train_length_method} | "
        f"wd={cfg.weight_decay} | mlp={int(cfg.use_mlp)} | layers={cfg.n_layers} | N={cfg.vocab_n} | "
        f"E={cfg.n_embd} ({cfg.emb_ratio_label}) | finalLN={int(cfg.use_final_LN)} | batch={cfg.effective_batch_size}"
    )

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    for itr in trange(cfg.max_iters, desc=f"train {cfg.train_order:03d}/{len(EXPERIMENTS)}", leave=True):
        current_length = select_train_length(cfg, itr, train_generator)
        lr = get_lr(itr, cfg)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        optimizer.zero_grad(set_to_none=True)
        do_log = ((itr + 1) % cfg.log_interval == 0) or (itr == 0) or (itr + 1 == cfg.max_iters)
        track_sample_acc_threshold = iter_to_sample_acc_gt_0_9999 is None
        token_correct = token_total = sample_correct = sample_total = 0

        idx = get_batch(
            batch_size=cfg.effective_batch_size,
            length=current_length,
            device=DEVICE,
            vocab_n=cfg.vocab_n,
            allow_duplicates=cfg.allow_duplicates,
            generator=train_generator,
        )
        with autocast_ctx(DEVICE, enabled=True):
            logits, loss = model(idx, block_size=current_length, return_full_logits=False)

        if do_log or track_sample_acc_threshold:
            with torch.no_grad():
                targets = idx[:, current_length + 1:]
                preds = logits.detach().argmax(dim=-1)
                sample_correct += int((preds == targets).all(dim=1).sum().item())
                sample_total += int(targets.size(0))
                if do_log:
                    token_correct += int((preds == targets).sum().item())
                    token_total += int(targets.numel())

        if scaler.is_enabled():
            scaler.scale(loss).backward()
        else:
            loss.backward()
        if scaler.is_enabled():
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()

        if track_sample_acc_threshold:
            current_sample_acc = float(sample_correct / max(sample_total, 1))
            if current_sample_acc > 0.9999:
                iter_to_sample_acc_gt_0_9999 = int(itr + 1)

        if do_log:
            row = {
                "iter": int(itr + 1),
                "lr": float(lr),
                "loss": float(loss.detach().float().item()),
                "token_acc": float(token_correct / max(token_total, 1)),
                "sample_acc": float(sample_correct / max(sample_total, 1)),
                "train_block_size": int(current_length),
            }
            if cfg.eval_during_training and eval_lengths:
                eval_metrics = evaluate_teacher_forced_lengths(model, cfg, eval_lengths, num_batches=1, generator_seed=int(cfg.seed + 100_000 + itr))
                row["eval_token_acc_by_length"] = {int(length): float(metrics["token_acc"]) for length, metrics in eval_metrics.items()}
                eval_summary = " | ".join([f"L={length}:{metrics['token_acc']:.4f}" for length, metrics in eval_metrics.items()])
            else:
                row["eval_token_acc_by_length"] = {}
                eval_summary = ""
            history.append(row)
            print(
                f"itr={itr + 1:6d} | lr={lr:.3e} | train_L={current_length:4d} | loss={row['loss']:.6f} | "
                f"token_acc={row['token_acc']:.4f} | sample_acc={row['sample_acc']:.4f}"
                + (f" | {eval_summary}" if eval_summary else "")
            )
            if cfg.early_stop_sample_accuracy_threshold is not None and row["sample_acc"] >= float(cfg.early_stop_sample_accuracy_threshold):
                print(f"[early stop] sample_acc={row['sample_acc']:.4f} reached threshold {float(cfg.early_stop_sample_accuracy_threshold):.4f} at iter {itr + 1}")
                break

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    training_minutes = (time.perf_counter() - t0) / 60.0
    if len(history) == 0:
        raise RuntimeError("Training history is empty; expected at least one logged row.")
    final_metrics = history[-1]
    during_training_plot = save_training_eval_plot(cfg, history) if cfg.eval_during_training else None

    summary_row = {
        "train_order": int(cfg.train_order),
        "model_id": str(cfg.model_id),
        "artifact_name": final_model_name(cfg),
        "seed": int(cfg.seed),
        "block_size": int(cfg.block_size),
        "train_length_method": str(cfg.train_length_method),
        "weight_decay": float(cfg.weight_decay),
        "use_mlp": bool(cfg.use_mlp),
        "n_layers": int(cfg.n_layers),
        "vocab_n": int(cfg.vocab_n),
        "model_vocab_size": int(cfg.vocab_n + 1),
        "embedding_size": int(cfg.n_embd),
        "embedding_ratio": str(cfg.emb_ratio_label),
        "use_final_LN": bool(cfg.use_final_LN),
        "allow_duplicates": bool(cfg.allow_duplicates),
        "without_pos": bool(cfg.without_pos),
        "n_heads": int(cfg.n_heads),
        "max_iters": int(cfg.max_iters),
        "max_seq_len": int(cfg.max_seq_len),
        "max_supported_length": int(cfg.max_supported_length),
        "eval_during_training": bool(cfg.eval_during_training),
        "eval_length_multipliers": json.dumps([float(x) for x in cfg.eval_length_multipliers]),
        "training_time_minutes": float(training_minutes),
        "final_training_loss": float(final_metrics["loss"]),
        "final_training_token_accuracy": float(final_metrics["token_acc"]),
        "final_training_sample_accuracy": float(final_metrics["sample_acc"]),
        "iter_to_sample_acc_gt_0_9999": iter_to_sample_acc_gt_0_9999,
        "during_training_eval_plot_path": during_training_plot,
        "artifact_path": str(artifact_path),
    }

    cpu_state_dict = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    artifact = {
        "artifact_type": "final_model",
        "train_config": asdict(cfg),
        "model_config": asdict(model_cfg),
        "model_state_dict": cpu_state_dict,
        "final_metrics": {
            "training_time_minutes": float(training_minutes),
            "final_training_loss": float(final_metrics["loss"]),
            "final_training_token_accuracy": float(final_metrics["token_acc"]),
            "final_training_sample_accuracy": float(final_metrics["sample_acc"]),
            "iter_to_sample_acc_gt_0_9999": iter_to_sample_acc_gt_0_9999,
        },
        "summary_row": summary_row,
        "history": history,
        "resolved_eval_lengths": [int(x) for x in eval_lengths],
        "during_training_eval_plot_path": during_training_plot,
    }
    torch.save(artifact, artifact_path)
    print(f"[saved final model] {artifact_path}")

    del cpu_state_dict, artifact, model, optimizer, scaler
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return summary_row



In [ ]:
# --- Google Drive folder setup + exact grid preview ---

try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    GDRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    GDRIVE_ROOT = Path.cwd() / "MyDrive"
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("[local fallback] Using", GDRIVE_ROOT)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_FOLDER_NAME = f"sortgpt_grid_train_{RUN_STAMP}"
RUN_DIR = GDRIVE_ROOT / RUN_FOLDER_NAME
MODELS_DIR = RUN_DIR / "final_models"
LENGTH_EVAL_DIR = RUN_DIR / "length_generalization"
DURING_TRAINING_EVAL_DIR = LENGTH_EVAL_DIR / "during_training"
POST_TRAINING_EVAL_DIR = LENGTH_EVAL_DIR / "post_training"
SUMMARY_CSV_PATH = RUN_DIR / "training_summary.csv"
MANIFEST_PATH = RUN_DIR / "run_manifest.json"

RUN_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
LENGTH_EVAL_DIR.mkdir(parents=True, exist_ok=True)
DURING_TRAINING_EVAL_DIR.mkdir(parents=True, exist_ok=True)
POST_TRAINING_EVAL_DIR.mkdir(parents=True, exist_ok=True)

FORCE_RETRAIN = False
EXPERIMENTS = build_experiment_specs(save_dir=str(MODELS_DIR), eval_dir=str(LENGTH_EVAL_DIR))

save_run_manifest(MANIFEST_PATH, run_dir=RUN_DIR, models_dir=MODELS_DIR, summary_csv_path=SUMMARY_CSV_PATH, experiments=EXPERIMENTS)

print("RUN_DIR               =", RUN_DIR)
print("MODELS_DIR            =", MODELS_DIR)
print("LENGTH_EVAL_DIR       =", LENGTH_EVAL_DIR)
print("DURING_TRAINING_DIR   =", DURING_TRAINING_EVAL_DIR)
print("POST_TRAINING_DIR     =", POST_TRAINING_EVAL_DIR)
print("SUMMARY_CSV_PATH      =", SUMMARY_CSV_PATH)
print("MANIFEST_PATH         =", MANIFEST_PATH)
print("NUM_MODELS            =", len(EXPERIMENTS))
display(pd.DataFrame([asdict(cfg) for cfg in EXPERIMENTS]).head(12))


In [ ]:
# --- Train the full grid, save final models, and keep the CSV updated ---

rows_by_id: Dict[str, Dict[str, Any]] = {}
if SUMMARY_CSV_PATH.exists():
    try:
        existing_df = pd.read_csv(SUMMARY_CSV_PATH)
        if "model_id" in existing_df.columns:
            rows_by_id = {str(row["model_id"]): row.to_dict() for _, row in existing_df.iterrows()}
            print(f"[resume] loaded {len(rows_by_id)} existing CSV rows")
    except Exception as e:
        print(f"[warn] Could not read existing CSV: {e}")

for cfg in EXPERIMENTS:
    row = train_sortgpt(cfg, force_retrain=FORCE_RETRAIN)
    rows_by_id[cfg.model_id] = row
    current_df = save_summary_csv(rows_by_id, SUMMARY_CSV_PATH)
    print(f"[csv updated] {SUMMARY_CSV_PATH}")
    display(current_df.tail(3))

final_df = save_summary_csv(rows_by_id, SUMMARY_CSV_PATH)
print("[done] all requested models processed")
print("Run folder:", RUN_DIR)
print("Final-model folder:", MODELS_DIR)
print("Length-generalization folder:", LENGTH_EVAL_DIR)
print("Summary CSV:", SUMMARY_CSV_PATH)
display(final_df)


In [ ]:
# --- Length-generalization evaluation / plotting after training ---

summary_df = pd.read_csv(SUMMARY_CSV_PATH) if SUMMARY_CSV_PATH.exists() else pd.DataFrame()
if summary_df.empty:
    print("[warn] No trained models found in the summary CSV yet.")
elif not LENGTH_GENERALIZATION_EVALS_ENABLED:
    print("[skip] Length-generalization evals are disabled because EARLY_STOP_SAMPLE_ACCURACY_THRESHOLD is not None.")
else:
    if EVAL_DURING_TRAINING:
        plot_rows: List[Dict[str, Any]] = []
        for cfg in EXPERIMENTS:
            artifact_path = final_model_path(cfg)
            if not artifact_path.exists():
                continue
            artifact = torch.load(artifact_path, map_location="cpu")
            loaded_cfg = TrainConfig(**artifact["train_config"])
            if loaded_cfg.early_stop_sample_accuracy_threshold is not None:
                continue
            history = artifact.get("history", [])
            saved_plot = artifact.get("during_training_eval_plot_path")
            final_eval = None
            for row in reversed(history):
                if row.get("eval_token_acc_by_length"):
                    final_eval = row["eval_token_acc_by_length"]
                    break
            plot_rows.append({
                "model_id": loaded_cfg.model_id,
                "train_length_method": loaded_cfg.train_length_method,
                "weight_decay": loaded_cfg.weight_decay,
                "plot_path": saved_plot,
                "final_logged_eval": json.dumps(final_eval or {}),
            })
        print("Saved during-training plots in:", DURING_TRAINING_EVAL_DIR)
        display(pd.DataFrame(plot_rows))

    eval_rows: List[Dict[str, Any]] = []
    for cfg in EXPERIMENTS:
        artifact_path = final_model_path(cfg)
        if not artifact_path.exists():
            print(f"[skip missing artifact] {artifact_path}")
            continue
        model, artifact, loaded_cfg = load_model_from_artifact(artifact_path)
        if loaded_cfg.early_stop_sample_accuracy_threshold is not None:
            del model
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            continue
        eval_lengths = artifact.get("resolved_eval_lengths") or resolve_eval_lengths(loaded_cfg.block_size, loaded_cfg.vocab_n, loaded_cfg.eval_length_multipliers)
        metrics = evaluate_teacher_forced_lengths(model, loaded_cfg, eval_lengths, num_batches=int(loaded_cfg.post_training_eval_batches), generator_seed=int(loaded_cfg.seed + 200_000))
        for length, result in metrics.items():
            eval_rows.append({
                "model_id": loaded_cfg.model_id,
                "block_size": int(length),
                "token_acc": float(result["token_acc"]),
                "loss": float(result["loss"]),
                "num_batches": int(result["num_batches"]),
                "train_length_method": loaded_cfg.train_length_method,
                "weight_decay": loaded_cfg.weight_decay,
                "base_k": loaded_cfg.block_size,
            })
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    eval_df = pd.DataFrame(eval_rows)
    if eval_df.empty:
        print("[warn] No post-training evaluation rows were produced.")
    else:
        csv_path = post_training_eval_csv_path()
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        eval_df.to_csv(csv_path, index=False)
        plt.figure(figsize=(12, 8))
        for model_id, group in eval_df.groupby("model_id"):
            group = group.sort_values("block_size")
            plt.plot(group["block_size"], group["token_acc"], marker="o", linewidth=1.2, label=model_id)
        plt.xlabel("Block size")
        plt.ylabel("Token accuracy")
        plt.title("Post-training length generalization")
        plt.grid(alpha=0.25)
        plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
        plt.tight_layout()
        plot_path = post_training_eval_plot_path()
        plt.savefig(plot_path, dpi=180, bbox_inches="tight")
        plt.close()
        print("Saved post-training eval CSV:", csv_path)
        print("Saved post-training eval plot:", plot_path)
        display(eval_df.sort_values(["model_id", "block_size"]))
